# 03. Feature Set: w2v_brand

**관점**: '무엇을 사는가' - brd_nm 임베딩 (고카디널리티)

이 노트북은 **하나의 feature set** 에 대해 5-fold 외부 CV로 AutoGluon(블랙박스 모델)을 학습하고, CV 지표(AUC/LogLoss/Brier)를 확인한 뒤 **OOF 예측**과 **test 예측**을 저장합니다.

- `oof_{setname}.csv` : train OOF (메타모델 학습 입력)
- `oof_test_{setname}.csv` : test 예측 평균 (메타모델 추론 입력)

> 모든 set 노트북은 동일한 `folds.csv` split 을 공유합니다 (스태킹 정합성).

## 0. 설치 & 라이브러리

In [3]:
import sys
!{sys.executable} -m pip install ipython-autotime
!{sys.executable} -m pip install autogluon.tabular gensim

Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable


In [5]:
%load_ext autotime

import os
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, log_loss, brier_score_loss
from autogluon.tabular import TabularPredictor
import warnings, logging
warnings.filterwarnings('ignore')
logging.getLogger('autogluon').setLevel(logging.ERROR)

pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 100)

# ---------------- 전역 설정 ----------------
SEED        = 42
N_SPLITS    = 5
TIME_LIMIT  = 300          # set당 fold 1개 학습 시간(초). 빠른 테스트 시 줄이세요.
PRESETS     = 'good_quality'
EVAL_METRIC = 'roc_auc'    # AutoGluon 내부 모델 선택 기준

DATA_DIR = '../../data/raw'
OOF_DIR  = '../../outputs/oof'
os.makedirs(OOF_DIR, exist_ok=True)

time: 8.64 s (started: 2026-06-05 18:29:32 +09:00)


## 1. 데이터 로드

In [7]:
# 거래 단위 원본 데이터 로드
X_train_trans = pd.read_csv(f'{DATA_DIR}/train_transactions.csv', encoding='cp949')
X_test_trans  = pd.read_csv(f'{DATA_DIR}/test_transactions.csv',  encoding='cp949')
y_train       = pd.read_csv(f'{DATA_DIR}/y_train.csv',            encoding='cp949')

print('train trans:', X_train_trans.shape, '| test trans:', X_test_trans.shape)
print('y_train:', y_train.shape)

train trans: (1036653, 16) | test trans: (689777, 16)
y_train: (30000, 2)
time: 15.2 s (started: 2026-06-05 18:29:40 +09:00)


## 2. Fold split 로드/생성 (공유)

In [9]:
# ===== Fold split: 모든 feature set / 노트북이 공유 =====
# 첫 실행에서 folds.csv 가 없으면 생성, 있으면 로드 -> 7개 set 모두 동일 split 사용 보장
FOLDS_PATH = f'{OOF_DIR}/folds.csv'

if os.path.exists(FOLDS_PATH):
    folds_df = pd.read_csv(FOLDS_PATH)
    print('기존 folds.csv 로드:', folds_df.shape)
else:
    base = y_train[['custid', 'gender']].sort_values('custid').reset_index(drop=True)
    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
    base['fold'] = -1
    for f, (_, val_idx) in enumerate(skf.split(base['custid'], base['gender'])):
        base.loc[val_idx, 'fold'] = f
    folds_df = base[['custid', 'fold']]
    folds_df.to_csv(FOLDS_PATH, index=False)
    print('folds.csv 신규 생성:', folds_df.shape)

print(folds_df['fold'].value_counts().sort_index())

기존 folds.csv 로드: (30000, 2)
fold
0    6000
1    6000
2    6000
3    6000
4    6000
Name: count, dtype: int64
time: 31 ms (started: 2026-06-05 18:29:56 +09:00)


## 3. Feature 생성  ← 팀원이 통째로 교체하는 영역
`features_train`, `features_test` 두 변수를 `custid` + feature 컬럼 형태로 만들면 됩니다.

In [11]:
SETNAME = 'w2v_brand'

time: 16 ms (started: 2026-06-05 18:29:56 +09:00)


In [12]:
# ===== FEATURE 생성 (수정/교체 가능): W2V_brd_nm =====
# 관점: '무엇을 사는가' 를 brd_nm 임베딩으로 표현 (순수 W2V, manual 미포함)
from gensim.models import Word2Vec

TOKEN_COL   = 'brd_nm'
VECTOR_SIZE = 64 # brd_nm 은 고카디널리티(브랜드 수천 개) -> VECTOR_SIZE = 128 로 올려보는 실험을 권장
WINDOW = 5; MIN_COUNT = 1; SG = 1; EPOCHS = 10

def train_w2v(df, token_col=TOKEN_COL):
    df = df.copy()
    df['sales_datetime'] = pd.to_datetime(df['sales_datetime'])
    df = df.sort_values(['custid','sales_datetime'])
    df[token_col] = df[token_col].astype(str)
    sentences = df.groupby('custid')[token_col].apply(list).tolist()
    return Word2Vec(sentences=sentences, vector_size=VECTOR_SIZE, window=WINDOW,
                    min_count=MIN_COUNT, sg=SG, workers=4, epochs=EPOCHS, seed=SEED)

def make_w2v_features(df, model, token_col=TOKEN_COL):
    df = df.copy()
    df['sales_datetime'] = pd.to_datetime(df['sales_datetime'])
    df = df.sort_values(['custid','sales_datetime'])
    df[token_col] = df[token_col].astype(str)
    vs = model.vector_size
    def avg_vec(tokens):
        vecs = [model.wv[t] for t in tokens if t in model.wv.key_to_index]
        return np.mean(vecs, axis=0) if vecs else np.zeros(vs)
    seq = df.groupby('custid')[token_col].apply(list)
    mat = np.vstack(seq.apply(avg_vec).values)
    cols = [f'{token_col}_w2v_{i}' for i in range(vs)]
    return pd.DataFrame(mat, columns=cols, index=seq.index).reset_index()

# W2V 는 train 으로만 학습 (test leakage 방지)
w2v_model = train_w2v(X_train_trans)
features_train = make_w2v_features(X_train_trans, w2v_model)
features_test  = make_w2v_features(X_test_trans,  w2v_model)
print('W2V_brd_nm features:', features_train.shape[1] - 1)

# ===== train/test feature 컬럼 정렬 (crosstab 류 카테고리 불일치 방지) =====
feat_cols_ref = [c for c in features_train.columns if c != 'custid']
# test 를 train 컬럼에 맞춤: train 에만 있는 건 0 채움, test 에만 있는 건 버림
features_test = features_test.reindex(columns=['custid'] + feat_cols_ref, fill_value=0)
print('정렬 후 feature 수:', len(feat_cols_ref))

W2V_brd_nm features: 64
정렬 후 feature 수: 64
time: 1min 1s (started: 2026-06-05 18:29:56 +09:00)


## 4. 5-fold 외부 CV (AutoGluon = 하나의 블랙박스 모델)

In [14]:
# ===== 5-fold 외부 CV: AutoGluon 을 하나의 블랙박스 모델로 취급 =====
# - 각 fold: train fold 로 AutoGluon fit -> valid fold 예측 = OOF (leakage 없음)
# - test: 5개 fold 모델 예측의 평균 = test meta feature
SETNAME = SETNAME  # 위에서 정의됨

# custid / target / fold 정렬 맞추기
data = features_train.merge(folds_df, on='custid', how='inner') \
                     .merge(y_train[['custid', 'gender']], on='custid', how='inner')
data = data.sort_values('custid').reset_index(drop=True)

feat_cols = [c for c in features_train.columns if c != 'custid']

oof_pred  = np.zeros(len(data))
test_pred = np.zeros(len(features_test))
test_sorted = features_test.sort_values('custid').reset_index(drop=True)

for f in range(N_SPLITS):
    tr_idx = data.index[data['fold'] != f]
    va_idx = data.index[data['fold'] == f]

    train_part = data.loc[tr_idx, feat_cols + ['gender']]
    valid_part = data.loc[va_idx, feat_cols]

    predictor = TabularPredictor(
        label='gender', problem_type='binary',
        eval_metric=EVAL_METRIC, verbosity=0,
    ).fit(train_data=train_part, presets=PRESETS, time_limit=TIME_LIMIT)

    pos = predictor.positive_class
    oof_pred[va_idx] = predictor.predict_proba(valid_part)[pos].values
    test_pred += predictor.predict_proba(test_sorted[feat_cols])[pos].values / N_SPLITS

    fold_auc = roc_auc_score(data.loc[va_idx, 'gender'], oof_pred[va_idx])
    print(f'[fold {f}] valid AUC = {fold_auc:.5f}')

[fold 0] valid AUC = 0.67879
[fold 1] valid AUC = 0.66977
[fold 2] valid AUC = 0.67174
[fold 3] valid AUC = 0.66502
[fold 4] valid AUC = 0.67847
time: 27min 16s (started: 2026-06-05 18:30:57 +09:00)


## 5. CV 지표

In [16]:
# ===== CV 지표: AUC(순위) + LogLoss/Brier(칼리브레이션·오답 페널티) =====
y_true = data['gender'].values
cv_auc   = roc_auc_score(y_true, oof_pred)
cv_ll    = log_loss(y_true, oof_pred)
cv_brier = brier_score_loss(y_true, oof_pred)

print(f'=== {SETNAME} CV ===')
print(f'AUC      : {cv_auc:.5f}   (순위 품질, 높을수록 좋음)')
print(f'LogLoss  : {cv_ll:.5f}   (확신한 오답에 큰 페널티, 낮을수록 좋음)')
print(f'Brier    : {cv_brier:.5f}   (확률 칼리브레이션, 낮을수록 좋음)')

=== w2v_brand CV ===
AUC      : 0.67240   (순위 품질, 높을수록 좋음)
LogLoss  : 0.57363   (확신한 오답에 큰 페널티, 낮을수록 좋음)
Brier    : 0.19495   (확률 칼리브레이션, 낮을수록 좋음)
time: 78 ms (started: 2026-06-05 18:58:14 +09:00)


## 6. OOF / test 예측 저장

In [24]:
# ===== OOF / test 예측 저장 (스키마: custid, pred_proba) =====
oof_out = pd.DataFrame({
    'custid': data['custid'].values,
    'pred_proba': oof_pred,
    'fold': data['fold'].values,
})
oof_out.to_csv(f'{OOF_DIR}/oof_{SETNAME}_0.csv', index=False)

test_out = pd.DataFrame({
    'custid': test_sorted['custid'].values,
    'pred_proba': test_pred,
})
test_out.to_csv(f'{OOF_DIR}/test_{SETNAME}_0.csv', index=False)

print('저장 완료:')
print(f'  {OOF_DIR}/oof_{SETNAME}_0.csv      ', oof_out.shape)
print(f'  {OOF_DIR}/oof_test_{SETNAME}_0.csv ', test_out.shape)
display(oof_out.head())

저장 완료:
  ../../outputs/oof/oof_w2v_brand.csv       (30000, 3)
  ../../outputs/oof/oof_test_w2v_brand.csv  (19995, 2)


,custid,pred_proba,fold
0,0,0.382588,3
1,1,0.321690,1
2,2,0.469630,0
3,3,0.349854,0
4,4,0.161385,0


time: 266 ms (started: 2026-06-05 18:58:45 +09:00)
